# Working Memory (Manage Tasks) Test

Calls `manage_tasks` directly with its injected `conversation_id` against a fresh conversation, and reads the stored `task_plan` back from the `sessions` document. Needs Firestore access (the same credentials the gateway uses).

In [2]:
import json
import sys

sys.path.insert(0, "..")

from app.gateway.gateway import create_conversation, fetch_session_memory
from app.orchestrator.tools import manage_tasks

In [4]:
USER_ID = "gateway-test-user"
conversation_id = await create_conversation(USER_ID)
print("conversation_id:", conversation_id)


async def save(**args):
    return await manage_tasks.ainvoke({**args, "conversation_id": conversation_id})


async def show_plan():
    memory = await fetch_session_memory(conversation_id)
    print(json.dumps(memory.task_plan, indent=1))
    print("history entries (should stay 0):", len(memory.history_messages))


def task(text, status="todo"):
    return {"text": text, "status": status}

conversation_id: d64c76b6-c8fe-4622-af2d-03e70a8cc525


## Save a plan

In [8]:
print(await save(
    scope="Model: XGBoost Ranker. Split: test. Bar charts sorted high to low.",
    tasks=[
        task("Reorder rate by department, as a bar chart with a two-sentence takeaway.", "in_progress"),
        task("Top 10 features by SHAP importance, as a bar chart with a short explanation."),
        task("Beeswarm for the most important feature, produce department only."),
    ]))
await show_plan()

Saved the task plan (0 of 3 done).
{
 "tasks": [
  {
   "status": "in_progress",
   "text": "Reorder rate by department, as a bar chart with a two-sentence takeaway."
  },
  {
   "status": "todo",
   "text": "Top 10 features by SHAP importance, as a bar chart with a short explanation."
  },
  {
   "status": "todo",
   "text": "Beeswarm for the most important feature, produce department only."
  }
 ],
 "scope": "Model: XGBoost Ranker. Split: test. Bar charts sorted high to low."
}
history entries (should stay 0): 0


## Update statuses without resending the scope (the scope should stay)

In [4]:
print(await save(tasks=[
    task("Reorder rate by department, as a bar chart with a two-sentence takeaway.", "done"),
    task("Top 10 features by SHAP importance, as a bar chart with a short explanation.", "in_progress"),
    task("Beeswarm for the most important feature, produce department only."),
]))
await show_plan()

Saved the task plan (1 of 3 done).
{
 "tasks": [
  {
   "status": "done",
   "text": "Reorder rate by department, as a bar chart with a two-sentence takeaway."
  },
  {
   "status": "in_progress",
   "text": "Top 10 features by SHAP importance, as a bar chart with a short explanation."
  },
  {
   "status": "todo",
   "text": "Beeswarm for the most important feature, produce department only."
  }
 ],
 "scope": "Model: XGBoost Ranker. Split: test. Bar charts sorted high to low."
}
history entries (should stay 0): 0


## Change the scope, then clear only the scope

In [5]:
tasks = [task("Reorder rate by department, as a bar chart.", "done")]
print(await save(tasks=tasks, scope="Model: LightGBM Classifier. Split: validation."))
await show_plan()
print(await save(tasks=tasks, scope=""))
await show_plan()

Saved the task plan (1 of 1 done).
{
 "tasks": [
  {
   "status": "done",
   "text": "Reorder rate by department, as a bar chart."
  }
 ],
 "scope": "Model: LightGBM Classifier. Split: validation."
}
history entries (should stay 0): 0
Saved the task plan (1 of 1 done).
{
 "tasks": [
  {
   "status": "done",
   "text": "Reorder rate by department, as a bar chart."
  }
 ],
 "scope": ""
}
history entries (should stay 0): 0


## Argument errors (each should come back as a readable message)

In [6]:
print(await save(tasks=[task("A", "in_progress"), task("B", "in_progress")]))
print(await save(tasks=[task("   ")]))
print(await save(tasks=[task("x" * 501)]))
print(await save(tasks=[task(f"Task {n}") for n in range(31)]))
print(await save(tasks=[task("A")], scope="s" * 801))

1 validation error for ManageTasksArgs
tasks
  Value error, More than one task is in progress; mark only the one you are working on. [type=value_error, input_value=[{'text': 'A', 'status': ...status': 'in_progress'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
1 validation error for ManageTasksArgs
tasks
  Value error, Task 1 is blank. [type=value_error, input_value=[{'text': '   ', 'status': 'todo'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
1 validation error for ManageTasksArgs
tasks
  Value error, Task 1 is 501 characters; keep each under 500. [type=value_error, input_value=[{'text': 'xxxxxxxxxxxxxx...xxx', 'status': 'todo'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
1 validation error for ManageTasksArgs
tasks
  Value error, A task list holds at most 30 tasks; merge or drop some. [type=value_error, input_value=[{'t

## Clear the whole plan

In [7]:
print(await save(tasks=[]))
await show_plan()

Cleared the task plan.
{}
history entries (should stay 0): 0


# Notes (manage_notes)

Calls `manage_notes` directly with its injected `conversation_id` against a fresh conversation, and reads the stored `notes` back from the `sessions` document after each step. Same Firestore access as above.

In [5]:
import asyncio

from app.orchestrator.tools import NOTE_MAX_CHARS, NOTES_MAX_COUNT, manage_notes

notes_conversation_id = await create_conversation(USER_ID)
print("conversation_id:", notes_conversation_id)


async def note(**args):
    return await manage_notes.ainvoke({**args, "conversation_id": notes_conversation_id})


async def stored_notes():
    return (await fetch_session_memory(notes_conversation_id)).notes


async def show_notes():
    memory = await fetch_session_memory(notes_conversation_id)
    for n in memory.notes:
        print(f'{n["id"]} ({n["kind"]}): {n["text"]}')
    if not memory.notes:
        print("(none)")
    print("history entries (should stay 0):", len(memory.history_messages))

conversation_id: eadf7678-96a3-495b-9a1a-47f8869721f1


## Add notes one at a time (ids should be 1, 2, 3)

In [6]:
print(await note(action="add", kind="goal", text="Understand what drives the reorder rate by department."))
print(await note(action="add", kind="preference", text="Show the SQL under every chart."))
print(await note(action="add", kind="definition", text="An active customer ordered in the last 30 days."))
await show_notes()

Added note 1.
Added note 2.
Added note 3.
1 (goal): Understand what drives the reorder rate by department.
2 (preference): Show the SQL under every chart.
3 (definition): An active customer ordered in the last 30 days.
history entries (should stay 0): 0


## Update replaces the whole note (note 2 changes, 1 and 3 stay)

In [7]:
print(await note(action="update", note_id=2, kind="constraint", text="Exclude the missing department."))
await show_notes()

Updated note 2.
1 (goal): Understand what drives the reorder rate by department.
2 (constraint): Exclude the missing department.
3 (definition): An active customer ordered in the last 30 days.
history entries (should stay 0): 0


## Delete, then add (ids are not renumbered; the new id should be 4)

In [8]:
print(await note(action="delete", note_id=1))
print(await note(action="add", kind="goal", text="Compare the top departments across both models."))
await show_notes()

Deleted note 1.
Added note 4.
2 (constraint): Exclude the missing department.
3 (definition): An active customer ordered in the last 30 days.
4 (goal): Compare the top departments across both models.
history entries (should stay 0): 0


## Batch of three adds (all three should land, with distinct ids)

In [9]:
before = len(await stored_notes())
results = await asyncio.gather(
    note(action="add", kind="preference", text="Round percentages to one decimal."),
    note(action="add", kind="constraint", text="Only use the test split."),
    note(action="add", kind="definition", text="A power user places 5 or more orders a month."),
)
print(results)
ids = [n["id"] for n in await stored_notes()]
print("added:", len(ids) - before, "| ids unique:", len(ids) == len(set(ids)))
await show_notes()

['Added note 5.', 'Added note 7.', 'Added note 6.']
added: 3 | ids unique: True
2 (constraint): Exclude the missing department.
3 (definition): An active customer ordered in the last 30 days.
4 (goal): Compare the top departments across both models.
5 (preference): Round percentages to one decimal.
6 (definition): A power user places 5 or more orders a month.
7 (constraint): Only use the test split.
history entries (should stay 0): 0


## Mixed batch: two deletes, an update and an add together (the deleted notes should stay gone)

In [10]:
ids = [n["id"] for n in await stored_notes()]
print("before:", ids)
results = await asyncio.gather(
    note(action="delete", note_id=ids[0]),
    note(action="delete", note_id=ids[1]),
    note(action="update", note_id=ids[2], kind="goal", text="Find which features separate the two models."),
    note(action="add", kind="preference", text="Keep answers under five sentences."),
)
print(results)
after = [n["id"] for n in await stored_notes()]
print("after:", after, "| deleted ids gone:", ids[0] not in after and ids[1] not in after)
await show_notes()

before: [2, 3, 4, 5, 6, 7]
['Deleted note 2.', 'Deleted note 3.', 'Updated note 4.', 'Added note 8.']
after: [4, 5, 6, 7, 8] | deleted ids gone: True
4 (goal): Find which features separate the two models.
5 (preference): Round percentages to one decimal.
6 (definition): A power user places 5 or more orders a month.
7 (constraint): Only use the test split.
8 (preference): Keep answers under five sentences.
history entries (should stay 0): 0


## Argument errors (each should come back as a readable message)

In [11]:
print(await note(action="add", text="No kind."))
print(await note(action="add", kind="goal"))
print(await note(action="update", kind="goal", text="No id."))
print(await note(action="delete"))
print(await note(action="add", kind="goal", text="   "))
print(await note(action="add", kind="goal", text="x" * (NOTE_MAX_CHARS + 1)))
print(await note(action="add", kind="mood", text="Unknown kind."))

1 validation error for ManageNotesArgs
  Value error, 'add' needs kind. [type=value_error, input_value={'action': 'add', 'text':...495b-9a1a-47f8869721f1'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
1 validation error for ManageNotesArgs
  Value error, 'add' needs text. [type=value_error, input_value={'action': 'add', 'kind':...495b-9a1a-47f8869721f1'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
1 validation error for ManageNotesArgs
  Value error, 'update' needs note_id. [type=value_error, input_value={'action': 'update', 'kin...495b-9a1a-47f8869721f1'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
1 validation error for ManageNotesArgs
  Value error, 'delete' needs note_id. [type=value_error, input_value={'action': 'delete', 'con...495b-9a1a-47f8869721f1'}, input_type=dict]
    For further information visit https://errors.

## Apply errors: an unknown id, then a full list (both should list the existing notes)

In [12]:
print(await note(action="update", note_id=99, kind="goal", text="No such note."))
print(await note(action="delete", note_id=99))
for n in range(len(await stored_notes()), NOTES_MAX_COUNT):
    await note(action="add", kind="goal", text=f"Filler note {n}.")
print(await note(action="add", kind="goal", text="One too many."))

Note 99 not found. Existing notes: 4 (goal): Find which features separate the two models.; 5 (preference): Round percentages to one decimal.; 6 (definition): A power user places 5 or more orders a month.; 7 (constraint): Only use the test split.; 8 (preference): Keep answers under five sentences.
Note 99 not found. Existing notes: 4 (goal): Find which features separate the two models.; 5 (preference): Round percentages to one decimal.; 6 (definition): A power user places 5 or more orders a month.; 7 (constraint): Only use the test split.; 8 (preference): Keep answers under five sentences.
A conversation holds at most 25 notes; delete or merge some. Existing notes: 4 (goal): Find which features separate the two models.; 5 (preference): Round percentages to one decimal.; 6 (definition): A power user places 5 or more orders a month.; 7 (constraint): Only use the test split.; 8 (preference): Keep answers under five sentences.; 9 (goal): Filler note 5.; 10 (goal): Filler note 6.; 11 (goal):